# Лабораторная работа 5. Нейросетевая классификация отзывов Кинопоиска

В этой работе классический pipeline из Lab IV заменяется на нейросетевой подход: токенизация, обучаемый `Embedding` и три архитектурных эксперимента на PyTorch. Датасет остается тем же, чтобы результаты можно было содержательно сравнивать с предыдущей лабораторной.

## Содержание

1. Окружение и настройки  
2. Загрузка датасета  
3. Разведочный анализ данных  
4. Предобработка текста  
5. Токенизация и padding  
6. Нейросетевые модели  
7. Обучение, calibration и сравнение качества  
8. Анализ ошибок  
9. Проверка на новых текстах  
10. Выводы

## 1. Окружение и настройки

В первой части подключаются библиотеки, фиксируются константы эксперимента и выбирается устройство. Ноутбук автоматически использует `cuda`, если доступна видеокарта, иначе переключается на `mps` или `cpu`.

### 1.1 Импорты

Импортируем инструменты для работы с таблицами, графиками, текстом и нейросетями. fastText не используется: по требованиям задания векторизация выполняется обучаемым слоем `Embedding` поверх токенизированных последовательностей.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import time
import tempfile
import warnings
from collections import Counter
from contextlib import nullcontext
from copy import deepcopy
from pathlib import Path
from typing import Iterable

MPLCONFIGDIR = Path(tempfile.gettempdir()) / "ml_lab_matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm
from wordcloud import WordCloud


try:
    from spacy.lang.ru.stop_words import STOP_WORDS as SPACY_RU_STOPWORDS
except Exception:
    SPACY_RU_STOPWORDS = {
        "и", "в", "во", "не", "что", "он", "на", "я", "с", "со", "как", "а", "то", "все",
        "она", "так", "его", "но", "да", "ты", "к", "у", "же", "вы", "за", "бы", "по", "только",
        "ее", "мне", "было", "вот", "от", "меня", "еще", "нет", "о", "из", "ему", "теперь",
        "когда", "даже", "ну", "вдруг", "ли", "если", "уже", "или", "ни", "быть", "был",
    }

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

### 1.2 Константы эксперимента

Здесь задаются основные ограничения. Размер словаря и длина последовательностей выбраны как практичный компромисс: достаточно большие для отзывов, но без чрезмерного роста времени обучения.

In [ ]:
SEED = 42
LABEL_ORDER = ["negative", "neutral", "positive"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_ORDER)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
NUM_CLASSES = len(LABEL_ORDER)

TEXT_COL = "review"
TARGET_COL = "sentiment"
PREPROCESS_MODE = "soft"  # "soft" для основного запуска, "lemma" как переключаемый baseline
PREPROCESS_VERSION = "v3_rating_train_bigrams"

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1
MIN_FREQ = 2
MAX_VOCAB_SIZE = 100_000
EMBEDDING_DIM = 200

USE_RATING_TOKENS = True
USE_FREQUENT_BIGRAMS = True
FREQUENT_BIGRAM_MIN_FREQ = 25
FREQUENT_BIGRAM_MAX_COUNT = 30_000

CNN_MAX_LEN = 768
RNN_MAX_LEN = 640
COMBINED_MAX_LEN = 768
CNN_BATCH_SIZE = 512
RNN_BATCH_SIZE = 256
COMBINED_BATCH_SIZE = 256

MAX_EPOCHS = 25
PATIENCE = 5
BALANCE_STRATEGY = "sampler"  # "sampler" или "loss_weights"; одновременно не используются
CLASS_WEIGHT_ALPHA = 0.25
LABEL_SMOOTHING = 0.02
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

RUN_SEED_ENSEMBLE = False
ENSEMBLE_SEEDS = (42, 77, 2026)

### 1.3 Воспроизводимость и устройство

Фиксируем seed и выбираем ускоритель. Mixed precision включается только на CUDA, потому что на CPU и MPS такой режим может быть нестабильным или бесполезным.

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed(SEED)
DEVICE = get_device()
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"

print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")

## 2. Загрузка датасета

Используется подготовленный CSV с отзывами Кинопоиска. Если файл еще не лежит в `LAB_V/data/dataset`, ноутбук создает hardlink из Lab IV, а если hardlink недоступен - обычную копию.

### 2.1 Пути к данным

Находим корень проекта и готовим путь к датасету. Эта ячейка не скачивает данные из интернета.

In [ ]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "LAB_V").exists() and (path / "LAB_IV").exists():
            return path
    raise FileNotFoundError("Не удалось найти корень проекта с LAB_IV и LAB_V")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "LAB_V" / "data" / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATASET_DIR / "kinopoisk_reviews_prepared.csv"
LAB4_DATA_PATH = PROJECT_ROOT / "LAB_IV" / "data" / "dataset" / "kinopoisk_reviews_prepared.csv"

if not DATA_PATH.exists():
    if not LAB4_DATA_PATH.exists():
        raise FileNotFoundError(f"Не найден исходный датасет: {LAB4_DATA_PATH}")
    try:
        os.link(LAB4_DATA_PATH, DATA_PATH)
        print(f"Создан hardlink: {DATA_PATH}")
    except OSError:
        shutil.copy2(LAB4_DATA_PATH, DATA_PATH)
        print(f"Создана копия: {DATA_PATH}")

print(DATA_PATH)

### 2.2 Чтение таблицы

Загружаем данные и оставляем только строки с текстом и корректной меткой тональности.

In [ ]:
df = pd.read_csv(DATA_PATH)
required_columns = {TEXT_COL, TARGET_COL}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"В датасете нет обязательных колонок: {sorted(missing_columns)}")

df = df[[TEXT_COL, TARGET_COL]].dropna().copy()
df[TARGET_COL] = df[TARGET_COL].astype(str)
df = df[df[TARGET_COL].isin(LABEL_ORDER)].reset_index(drop=True)

display(df.head())
df.info()

## 3. Разведочный анализ данных

EDA проводится на полном датасете. Для отчета оставлены только самые полезные графики: распределение классов, длины отзывов и частотные слова.

### 3.1 Размеры отзывов

Считаем число символов и слов в каждом отзыве. Эти статистики дальше используются для выбора `max_len`.

In [ ]:
eda_df = df.copy()
eda_df["char_len"] = eda_df[TEXT_COL].astype(str).str.len()
eda_df["tokens_raw"] = eda_df[TEXT_COL].astype(str).str.split()
eda_df["word_len"] = eda_df["tokens_raw"].str.len()
eda_df["unique_word_len"] = eda_df["tokens_raw"].map(lambda tokens: len(set(tokens)))

length_summary = eda_df[["char_len", "word_len", "unique_word_len"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95]).T
unique_words = len(Counter(" ".join(eda_df[TEXT_COL].astype(str).str.lower()).split()))

print(f"Rows: {len(eda_df):,}")
print(f"Unique raw tokens: {unique_words:,}")
display(length_summary)

### 3.2 Распределение классов

Проверяем дисбаланс классов. В обучении он компенсируется весами в `CrossEntropyLoss`, а не удалением части данных.

In [ ]:
class_counts = eda_df[TARGET_COL].value_counts().reindex(LABEL_ORDER).reset_index()
class_counts.columns = ["sentiment", "count"]
class_counts["share"] = class_counts["count"] / class_counts["count"].sum()

display(class_counts)
fig = px.bar(
    class_counts,
    x="sentiment",
    y="count",
    text="count",
    title="Распределение классов",
    labels={"sentiment": "Класс", "count": "Количество"},
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(height=420)
fig.show()

### 3.3 Распределение длин

Гистограмма показывает, какую часть начала и конца отзыва имеет смысл сохранять. Для длинных рецензий используется head-tail truncation: начало часто задает контекст, а финальная оценка нередко находится в конце.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Символы", "Слова"))
fig.add_histogram(x=eda_df["char_len"], nbinsx=80, name="Символы", row=1, col=1)
fig.add_histogram(x=eda_df["word_len"], nbinsx=80, name="Слова", row=1, col=2)
fig.update_xaxes(title_text="Длина", row=1, col=1)
fig.update_xaxes(title_text="Длина", row=1, col=2)
fig.update_yaxes(title_text="Количество", row=1, col=1)
fig.update_layout(title="Распределение длины отзывов", height=430, showlegend=False)
fig.show()

### 3.4 Частотные слова

Смотрим наиболее частые токены после грубой очистки. Это помогает оценить шум и выбрать размер словаря для токенизатора.

In [ ]:
EDA_STOPWORDS = set(SPACY_RU_STOPWORDS) | {"это", "все", "ещё", "еще", "который", "фильм"}

raw_tokens = []
for text in eda_df[TEXT_COL].astype(str).sample(min(len(eda_df), 50_000), random_state=SEED):
    raw_tokens.extend(
        token for token in re.findall(r"[а-яa-zё]+", text.lower())
        if len(token) > 2 and token not in EDA_STOPWORDS
    )

top_words = pd.DataFrame(Counter(raw_tokens).most_common(25), columns=["token", "count"])
display(top_words.head(10))
fig = px.bar(top_words, x="count", y="token", orientation="h", title="Топ-25 частотных слов")
fig.update_layout(height=620, yaxis={"categoryorder": "total ascending"})
fig.show()

## 4. Предобработка текста

Основной режим `soft` сохраняет словоформы, важные для тональности. Режим `lemma` оставлен как переключаемый baseline, но для нейросетей чаще полезнее мягкая очистка без агрессивной нормализации.

### 4.1 Стоп-слова и тональные маркеры

Из стоп-слов исключаются отрицания и слова-усилители: для sentiment-классификации они несут прямой смысл.

In [ ]:
IMPORTANT_WORDS = {
    "не", "нет", "ни", "но", "никогда", "без",
    "очень", "слишком", "совсем", "почти",
    "плохо", "хорошо", "ужасно", "прекрасно", "отлично",
    "скучно", "интересно", "лучше", "хуже",
}
NEGATION_WORDS = {"не", "нет", "ни", "никогда", "без"}
STOPWORDS = set(SPACY_RU_STOPWORDS) - IMPORTANT_WORDS

### 4.2 Опциональная загрузка spaCy

spaCy нужен только для режима `lemma`. Если модель `ru_core_news_sm` не установлена, основной `soft`-режим продолжит работать.

In [ ]:
nlp = None


def get_spacy_model():
    global nlp
    if nlp is not None:
        return nlp
    try:
        import spacy
        nlp = spacy.load("ru_core_news_sm", disable=["parser", "ner"])
    except Exception as exc:
        raise RuntimeError(
            "Для PREPROCESS_MODE='lemma' нужна spaCy-модель ru_core_news_sm. "
            "Оставьте PREPROCESS_MODE='soft' или установите модель отдельно."
        ) from exc
    return nlp

### 4.3 Функции очистки текста

`normalize_text` приводит текст к единому виду. Rating-токены извлекаются только из самого отзыва до удаления цифр: например, `2 из 10` превращается в `rating_2_10 rating_low`. Это не использует метаданные датасета и не читает validation/test при построении словаря.

Negation bigrams строятся только по соседним токенам исходной строки, поэтому `не понравился` дает `не_понравился`, а `не менее 7 лет` не превращается в ложный признак `не_лет`. Частотные биграммы подключаются позже: их список строится только по train-части.

In [ ]:
RAW_TOKEN_RE = re.compile(r"[а-яa-z]+|\d+|[^\s]")
WORD_TOKEN_RE = re.compile(r"^[а-яa-z]+$")
RATING_RE = re.compile(r"(?<!\d)(10|[0-9])\s*(?:/|из)\s*10(?!\d)")
BOUNDARY_TOKEN = "<BOUNDARY>"
FREQUENT_BIGRAMS: set[tuple[str, str]] = set()
BIGRAM_HASH = "no_train_bigrams"


def normalize_text(text: str) -> str:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"[^а-яa-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_raw_text(text: str) -> list[str]:
    text = str(text).lower().replace("ё", "е")
    return RAW_TOKEN_RE.findall(text)


def is_word_token(token: str) -> bool:
    return bool(WORD_TOKEN_RE.fullmatch(token))


def keep_preprocessed_token(token: str) -> bool:
    return is_word_token(token) and len(token) >= 2 and token not in STOPWORDS


def rating_bucket(score: int) -> str:
    if score <= 4:
        return "rating_low"
    if score <= 6:
        return "rating_mid"
    return "rating_high"


def extract_rating_tokens(text: str) -> list[str]:
    if not USE_RATING_TOKENS:
        return []
    tokens: list[str] = []
    normalized = str(text).lower().replace("ё", "е")
    for match in RATING_RE.finditer(normalized):
        score = int(match.group(1))
        tokens.extend([f"rating_{score}_10", rating_bucket(score)])
    return tokens


def iter_adjacent_word_pairs(tokens: list[str]):
    for left, right in zip(tokens, tokens[1:]):
        if is_word_token(left) and is_word_token(right) and len(left) >= 2 and len(right) >= 2:
            yield left, right


def build_frequent_bigrams(
    texts: Iterable[str],
    min_freq: int = FREQUENT_BIGRAM_MIN_FREQ,
    max_count: int = FREQUENT_BIGRAM_MAX_COUNT,
) -> tuple[set[tuple[str, str]], Counter]:
    counter: Counter = Counter()
    for text in tqdm(list(texts), desc="train-bigrams"):
        for left, right in iter_adjacent_word_pairs(tokenize_raw_text(text)):
            if left in NEGATION_WORDS:
                continue
            if left in STOPWORDS and right in STOPWORDS:
                continue
            counter[(left, right)] += 1
    selected = {
        pair for pair, count in counter.most_common(max_count)
        if count >= min_freq
    }
    return selected, counter


def add_context_bigrams(tokens: list[str]) -> list[str]:
    enriched: list[str] = []
    for idx, token in enumerate(tokens):
        if keep_preprocessed_token(token):
            enriched.append(token)
        if idx + 1 >= len(tokens):
            continue
        next_token = tokens[idx + 1]
        if token in NEGATION_WORDS and keep_preprocessed_token(next_token):
            enriched.append(f"{token}_{next_token}")
        if USE_FREQUENT_BIGRAMS and (token, next_token) in FREQUENT_BIGRAMS:
            enriched.append(f"{token}_{next_token}")
    return enriched


def preprocess_text_soft(text: str) -> str:
    tokens = extract_rating_tokens(text)
    tokens.extend(add_context_bigrams(tokenize_raw_text(text)))
    return " ".join(tokens)


def preprocess_text_lemma(text: str) -> str:
    doc = get_spacy_model()(str(text).lower().replace("ё", "е"))
    tokens: list[str] = []
    for token in doc:
        if not token.is_alpha:
            tokens.append(BOUNDARY_TOKEN)
            continue
        lemma = re.sub(r"[^а-яa-z]", "", token.lemma_.lower().replace("ё", "е"))
        tokens.append(lemma or token.text.lower().replace("ё", "е"))
    result_tokens = extract_rating_tokens(text)
    result_tokens.extend(add_context_bigrams(tokens))
    return " ".join(result_tokens)


def preprocess_text(text: str, mode: str = PREPROCESS_MODE) -> str:
    if mode == "soft":
        return preprocess_text_soft(text)
    if mode == "lemma":
        return preprocess_text_lemma(text)
    raise ValueError("PREPROCESS_MODE должен быть 'soft' или 'lemma'")


def preprocess_texts(texts: Iterable[str], mode: str = PREPROCESS_MODE) -> list[str]:
    return [preprocess_text(text, mode=mode) for text in tqdm(list(texts), desc=f"preprocess:{mode}")]

### 4.4 Smoke-test preprocessing

Проверяем два риска preprocessing: отрицание не должно склеиваться со словом, которое стало соседним только после удаления стоп-слов, цифр или пунктуации; rating-токены должны извлекаться только из явных оценок внутри текста отзыва.

In [ ]:
negation_smoke_cases = {
    "Не понравился фильм": ("не_понравился", True),
    "Не менее 7 лет назад фильм смотрелся лучше": ("не_лет", False),
    "Не, фильм на самом деле хороший": ("не_фильм", False),
    "Без сюжета и без интересных героев": ("без_сюжета", True),
}

for sample_text, (token, should_exist) in negation_smoke_cases.items():
    processed = preprocess_text_soft(sample_text)
    assert (token in processed.split()) is should_exist, (sample_text, token, processed)

rating_smoke_cases = {
    "Фильм на 2 из 10, пересматривать не буду": {"rating_2_10", "rating_low"},
    "Оценка 6/10: местами хорошо, местами скучно": {"rating_6_10", "rating_mid"},
    "Это честные 9 из 10": {"rating_9_10", "rating_high"},
}

for sample_text, expected_tokens in rating_smoke_cases.items():
    processed_tokens = set(preprocess_text_soft(sample_text).split())
    assert expected_tokens <= processed_tokens, (sample_text, expected_tokens, processed_tokens)

print("Preprocessing smoke-tests passed")

### 4.4 Быстрая проверка предобработки

Проверяем на одном отзыве, что очистка не удаляет отрицания и добавляет negation-биграммы.

In [ ]:
example_text = df.loc[0, TEXT_COL]
print("Исходный текст:")
print(example_text[:500])
print("\nПосле предобработки:")
print(preprocess_text(example_text)[:500])

### 4.5 Train/validation/test split

Делим полный датасет на train/test 80/20, затем train/validation 85/15. Распределение классов везде остается естественным, без balanced subset.

In [ ]:
raw_train_df, test_raw_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df[TARGET_COL],
)
train_raw_df, val_raw_df = train_test_split(
    raw_train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=raw_train_df[TARGET_COL],
)

train_raw_df = train_raw_df.copy()
val_raw_df = val_raw_df.copy()
test_raw_df = test_raw_df.copy()
train_raw_df["split"] = "train"
val_raw_df["split"] = "val"
test_raw_df["split"] = "test"

model_raw_df = pd.concat([train_raw_df, val_raw_df, test_raw_df], ignore_index=True)

display(model_raw_df["split"].value_counts().rename_axis("split").to_frame("rows"))
display(pd.crosstab(model_raw_df["split"], model_raw_df[TARGET_COL]))

### 4.6 Train-only частотные биграммы

Частотные биграммы строятся только по `train_raw_df`, уже после split. Validation и test получают тот же фиксированный набор биграмм, поэтому здесь нет подсматривания в отложенные части датасета.

In [ ]:
if USE_FREQUENT_BIGRAMS:
    FREQUENT_BIGRAMS, train_bigram_counter = build_frequent_bigrams(train_raw_df[TEXT_COL])
    BIGRAM_HASH = hashlib.md5(
        "\n".join(f"{left}_{right}" for left, right in sorted(FREQUENT_BIGRAMS)).encode("utf-8")
    ).hexdigest()[:12]
else:
    FREQUENT_BIGRAMS, train_bigram_counter = set(), Counter()
    BIGRAM_HASH = "no_train_bigrams"

print(f"Frequent train bigrams: {len(FREQUENT_BIGRAMS):,}")
print(f"Bigram hash: {BIGRAM_HASH}")
display(pd.DataFrame(
    [(f"{left}_{right}", count) for (left, right), count in train_bigram_counter.most_common(15)],
    columns=["bigram", "train_count"],
))

### 4.7 Clean-cache

Предобработка полного корпуса занимает время, поэтому очищенный текст сохраняется в CSV-cache с метаданными. Cache привязан к версии preprocessing и train-only bigram hash, чтобы не переиспользовать старые признаки.

In [ ]:
CLEAN_CACHE_PATH = DATASET_DIR / f"kinopoisk_reviews_cleaned_{PREPROCESS_MODE}_{PREPROCESS_VERSION}_{BIGRAM_HASH}.csv"
CLEAN_META_PATH = DATASET_DIR / f"kinopoisk_reviews_cleaned_{PREPROCESS_MODE}_{PREPROCESS_VERSION}_{BIGRAM_HASH}.meta.json"


def clean_cache_is_valid(expected_df: pd.DataFrame) -> bool:
    if not CLEAN_CACHE_PATH.exists() or not CLEAN_META_PATH.exists():
        return False
    try:
        meta = json.loads(CLEAN_META_PATH.read_text())
    except json.JSONDecodeError:
        return False
    return (
        meta.get("mode") == PREPROCESS_MODE
        and meta.get("preprocess_version") == PREPROCESS_VERSION
        and meta.get("bigram_hash") == BIGRAM_HASH
        and meta.get("use_rating_tokens") == USE_RATING_TOKENS
        and meta.get("rows") == len(expected_df)
        and meta.get("label_counts") == expected_df[TARGET_COL].value_counts().sort_index().to_dict()
    )

### 4.8 Запуск или чтение предобработки

Если подходящий cache уже есть, читаем его. Иначе очищаем все тексты и сохраняем результат в `cleaned_text`.

In [ ]:
if clean_cache_is_valid(model_raw_df):
    model_df = pd.read_csv(CLEAN_CACHE_PATH)
    print(f"Loaded clean cache: {CLEAN_CACHE_PATH}")
else:
    model_df = model_raw_df.copy()
    model_df["cleaned_text"] = preprocess_texts(model_df[TEXT_COL], mode=PREPROCESS_MODE)
    model_df.to_csv(CLEAN_CACHE_PATH, index=False)
    CLEAN_META_PATH.write_text(json.dumps({
        "mode": PREPROCESS_MODE,
        "preprocess_version": PREPROCESS_VERSION,
        "bigram_hash": BIGRAM_HASH,
        "use_rating_tokens": USE_RATING_TOKENS,
        "frequent_bigram_count": len(FREQUENT_BIGRAMS),
        "rows": len(model_df),
        "label_counts": model_raw_df[TARGET_COL].value_counts().sort_index().to_dict(),
    }, ensure_ascii=False, indent=2))
    print(f"Saved clean cache: {CLEAN_CACHE_PATH}")

model_df["cleaned_text"] = model_df["cleaned_text"].fillna("")
display(model_df[["split", TARGET_COL, "cleaned_text"]].head())

## 5. Токенизация и padding

Словарь строится только по train-части, чтобы не подсматривать в validation и test. В него попадают обычные токены, rating-токены и train-only биграммы. Для длинных отзывов применяется head-tail truncation.

### 5.1 Финальные split-таблицы

Отделяем train, validation и test после предобработки.

In [ ]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"] == "val"].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))
display(pd.crosstab(model_df["split"], model_df[TARGET_COL], normalize="index"))

### 5.2 Vocabulary по train

Токенизатор реализован через `Counter`: это прямой аналог `Tokenizer`, но без привязки к TensorFlow/Keras.

In [ ]:
def iter_tokens(texts: Iterable[str]):
    for text in texts:
        yield from str(text).split()


def build_vocab(texts: Iterable[str], max_vocab_size: int = MAX_VOCAB_SIZE, min_freq: int = MIN_FREQ):
    counter = Counter(iter_tokens(texts))
    tokens = [token for token, freq in counter.most_common(max_vocab_size - 2) if freq >= min_freq]
    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    vocab.update({token: idx for idx, token in enumerate(tokens, start=2)})
    return vocab, counter


vocab, token_counter = build_vocab(train_df["cleaned_text"])
vocab_hash = hashlib.md5("\n".join(vocab.keys()).encode("utf-8")).hexdigest()[:12]

print(f"Vocab size: {len(vocab):,}")
print(f"Vocab hash: {vocab_hash}")
display(pd.DataFrame(token_counter.most_common(20), columns=["token", "count"]))

### 5.3 Обоснование `max_len`

Считаем длины очищенных train-текстов и сравниваем их с выбранными лимитами для CNN/RNN. CNN получает более длинный контекст, RNN ограничена сильнее из-за последовательной природы GRU.

In [ ]:
train_lengths = train_df["cleaned_text"].str.split().str.len()
length_stats = train_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("train_tokens")
display(length_stats)
print(f"CNN_MAX_LEN={CNN_MAX_LEN}, RNN_MAX_LEN={RNN_MAX_LEN}, COMBINED_MAX_LEN={COMBINED_MAX_LEN}")

### 5.4 Кодирование текстов

`truncate_head_tail` сохраняет начало и конец отзыва. Это важнее простого обрезания по началу, потому что итоговая оценка часто формулируется в финальных предложениях.

In [ ]:
def truncate_head_tail(ids: list[int], max_len: int) -> list[int]:
    if len(ids) <= max_len:
        return ids
    head_len = max_len // 2
    tail_len = max_len - head_len
    return ids[:head_len] + ids[-tail_len:]


def encode_one(text: str, vocab: dict[str, int], max_len: int) -> tuple[list[int], int]:
    ids = [vocab.get(token, UNK_IDX) for token in str(text).split()]
    if not ids:
        ids = [UNK_IDX]
    ids = truncate_head_tail(ids, max_len)
    length = len(ids)
    padded = ids + [PAD_IDX] * (max_len - length)
    return padded, length


def encode_texts(texts: Iterable[str], vocab: dict[str, int], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    encoded = [encode_one(text, vocab, max_len) for text in tqdm(list(texts), desc=f"encode:{max_len}")]
    input_ids = torch.tensor([row[0] for row in encoded], dtype=torch.long)
    lengths = torch.tensor([row[1] for row in encoded], dtype=torch.long)
    return input_ids, lengths

### 5.5 Encoded-cache

Закодированные тензоры сохраняются отдельно для каждого split, режима предобработки, словаря и `max_len`.

In [ ]:
ENCODED_CACHE_DIR = DATASET_DIR / "encoded_cache"
ENCODED_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def encoded_cache_path(split: str, max_len: int) -> Path:
    return ENCODED_CACHE_DIR / f"encoded_{PREPROCESS_MODE}_{PREPROCESS_VERSION}_{BIGRAM_HASH}_{vocab_hash}_{split}_{max_len}.pt"


def labels_to_tensor(labels: Iterable[str]) -> torch.Tensor:
    return torch.tensor([LABEL_TO_ID[label] for label in labels], dtype=torch.long)


def load_or_encode_split(split: str, split_df: pd.DataFrame, max_len: int) -> dict[str, torch.Tensor]:
    path = encoded_cache_path(split, max_len)
    if path.exists():
        return torch.load(path, map_location="cpu", weights_only=False)

    input_ids, lengths = encode_texts(split_df["cleaned_text"], vocab, max_len)
    payload = {
        "input_ids": input_ids,
        "lengths": lengths,
        "labels": labels_to_tensor(split_df[TARGET_COL]),
        "meta": {
            "split": split,
            "max_len": max_len,
            "mode": PREPROCESS_MODE,
            "preprocess_version": PREPROCESS_VERSION,
            "bigram_hash": BIGRAM_HASH,
            "vocab_hash": vocab_hash,
        },
    }
    torch.save(payload, path)
    return payload

### 5.6 Подготовка encoded tensors

Готовим разные длины последовательностей для CNN, RNN и комбинированной модели.

In [ ]:
split_frames = {"train": train_df, "val": val_df, "test": test_df}
max_lens = sorted({CNN_MAX_LEN, RNN_MAX_LEN, COMBINED_MAX_LEN})
encoded_by_max_len = {
    max_len: {split: load_or_encode_split(split, split_df, max_len) for split, split_df in split_frames.items()}
    for max_len in max_lens
}

for max_len, splits in encoded_by_max_len.items():
    print(max_len, {split: tuple(payload["input_ids"].shape) for split, payload in splits.items()})

### 5.7 Dataset и DataLoader

Dataset возвращает `input_ids`, реальные длины и метки. Для Windows `num_workers` автоматически ставится в `0`, чтобы избежать падений multiprocessing в ноутбуке.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, encoded_split: dict[str, torch.Tensor]):
        self.input_ids = encoded_split["input_ids"]
        self.lengths = encoded_split["lengths"]
        self.labels = encoded_split["labels"]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[idx],
            "lengths": self.lengths[idx],
            "labels": self.labels[idx],
        }


def resolve_dataloader_workers() -> int:
    if platform.system().lower().startswith("win"):
        return 0
    return min(4, os.cpu_count() or 0)


def make_weighted_sampler(labels: torch.Tensor) -> WeightedRandomSampler:
    counts = torch.bincount(labels, minlength=NUM_CLASSES).float()
    class_weights_for_sampling = labels.numel() / (NUM_CLASSES * counts.clamp_min(1))
    sample_weights = class_weights_for_sampling[labels]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


def make_loader(
    encoded_split: dict[str, torch.Tensor],
    batch_size: int,
    shuffle: bool = False,
    use_weighted_sampler: bool = False,
) -> DataLoader:
    workers = resolve_dataloader_workers()
    sampler = make_weighted_sampler(encoded_split["labels"]) if use_weighted_sampler else None
    return DataLoader(
        ReviewDataset(encoded_split),
        batch_size=batch_size,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=workers,
        pin_memory=PIN_MEMORY,
        persistent_workers=workers > 0,
    )

### 5.8 DataLoader для каждой архитектуры

Каждая модель получает свой `max_len` и batch size. Это позволяет держать RNN легче, не ухудшая CNN.

In [ ]:
def make_loaders(max_len: int, batch_size: int) -> dict[str, DataLoader]:
    encoded = encoded_by_max_len[max_len]
    use_sampler = BALANCE_STRATEGY == "sampler"
    return {
        "train": make_loader(encoded["train"], batch_size=batch_size, shuffle=True, use_weighted_sampler=use_sampler),
        "val": make_loader(encoded["val"], batch_size=batch_size * 2, shuffle=False),
        "test": make_loader(encoded["test"], batch_size=batch_size * 2, shuffle=False),
    }


loaders_by_model = {
    "TrainableCNN": make_loaders(CNN_MAX_LEN, CNN_BATCH_SIZE),
    "TrainablePackedBiGRU": make_loaders(RNN_MAX_LEN, RNN_BATCH_SIZE),
    "TrainableCNNBiGRU": make_loaders(COMBINED_MAX_LEN, COMBINED_BATCH_SIZE),
}

## 6. Нейросетевые модели

Все модели используют обучаемый `nn.Embedding`. Эксперименты отличаются тем, как извлекаются признаки из последовательности: свертки, рекуррентный контекст или обе ветки вместе.

### 6.1 Pooling helpers

Masked pooling нужен для RNN: он усредняет и максимизирует только реальные токены, не учитывая PAD.

In [ ]:
def sequence_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
    return torch.arange(max_len, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)


def masked_mean_pool(outputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = sequence_mask(lengths, outputs.size(1)).unsqueeze(-1).to(outputs.dtype)
    summed = (outputs * mask).sum(dim=1)
    denom = lengths.clamp_min(1).unsqueeze(1).to(outputs.dtype)
    return summed / denom


def masked_max_pool(outputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = sequence_mask(lengths, outputs.size(1)).unsqueeze(-1)
    fill_value = torch.finfo(outputs.dtype).min
    return outputs.masked_fill(~mask, fill_value).max(dim=1).values


class AttentionPool(nn.Module):
    def __init__(self, feature_dim: int):
        super().__init__()
        self.score = nn.Linear(feature_dim, 1)

    def forward(self, outputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        mask = sequence_mask(lengths, outputs.size(1))
        scores = self.score(outputs).squeeze(-1)
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (outputs * weights).sum(dim=1)


def count_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

### 6.2 Multi-kernel CNN

CNN ловит локальные n-граммы разной длины. `max pooling` сохраняет самый сильный сигнал, а `mean pooling` добавляет информацию о среднем фоне отзыва.

In [ ]:
class TextCNNMulti(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = EMBEDDING_DIM,
        num_classes: int = NUM_CLASSES,
        pad_idx: int = PAD_IDX,
        num_filters: int = 192,
        kernel_sizes: tuple[int, ...] = (1, 2, 3, 4, 5, 7, 9),
        embedding_dropout: float = 0.15,
        dropout: float = 0.45,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(embedding_dropout)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, num_filters, kernel_size=kernel_size, padding=kernel_size // 2)
            for kernel_size in kernel_sizes
        ])
        feature_dim = len(kernel_sizes) * num_filters * 2
        self.norm = nn.LayerNorm(feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor | None = None) -> torch.Tensor:
        x = self.embedding_dropout(self.embedding(input_ids)).transpose(1, 2)
        features = []
        for conv in self.convs:
            y = F.relu(conv(x))
            features.append(F.adaptive_max_pool1d(y, 1).squeeze(-1))
            features.append(F.adaptive_avg_pool1d(y, 1).squeeze(-1))
        return self.classifier(self.norm(torch.cat(features, dim=1)))

### 6.3 Packed BiGRU

Рекуррентная модель читает последовательность с учетом порядка слов. `pack_padded_sequence` убирает лишнюю работу по PAD-токенам, а финальные признаки собираются из hidden state, mean pooling и max pooling.

In [ ]:
class TextPackedRNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = EMBEDDING_DIM,
        hidden_size: int = 256,
        num_classes: int = NUM_CLASSES,
        pad_idx: int = PAD_IDX,
        rnn_type: str = "gru",
        num_layers: int = 1,
        embedding_dropout: float = 0.15,
        dropout: float = 0.45,
    ):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        rnn_cls = nn.GRU if self.rnn_type == "gru" else nn.LSTM
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(embedding_dropout)
        self.rnn = rnn_cls(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.25 if num_layers > 1 else 0.0,
        )
        rnn_out_dim = hidden_size * 2
        self.attention_pool = AttentionPool(rnn_out_dim)
        feature_dim = rnn_out_dim * 4
        self.norm = nn.LayerNorm(feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding_dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, hidden = self.rnn(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=input_ids.size(1))

        if isinstance(hidden, tuple):
            hidden = hidden[0]
        final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        lengths_on_device = lengths.to(outputs.device)
        features = torch.cat([
            final_hidden,
            masked_mean_pool(outputs, lengths_on_device),
            masked_max_pool(outputs, lengths_on_device),
            self.attention_pool(outputs, lengths_on_device),
        ], dim=1)
        return self.classifier(self.norm(features))

### 6.4 Комбинированная CNN+BiGRU

Комбинированная модель извлекает локальные фразы CNN-веткой и последовательный контекст GRU-веткой, затем объединяет признаки в общий классификатор.

In [ ]:
class TextCNNBiGRU(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = EMBEDDING_DIM,
        num_classes: int = NUM_CLASSES,
        pad_idx: int = PAD_IDX,
        cnn_filters: int = 128,
        kernel_sizes: tuple[int, ...] = (1, 2, 3, 4, 5, 7),
        gru_hidden: int = 192,
        embedding_dropout: float = 0.15,
        dropout: float = 0.5,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(embedding_dropout)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, cnn_filters, kernel_size=kernel_size, padding=kernel_size // 2)
            for kernel_size in kernel_sizes
        ])
        self.gru = nn.GRU(embedding_dim, gru_hidden, batch_first=True, bidirectional=True)
        rnn_out_dim = gru_hidden * 2
        self.attention_pool = AttentionPool(rnn_out_dim)

        cnn_dim = len(kernel_sizes) * cnn_filters * 2
        rnn_dim = rnn_out_dim * 4
        feature_dim = cnn_dim + rnn_dim
        self.norm = nn.LayerNorm(feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding_dropout(self.embedding(input_ids))

        cnn_input = embedded.transpose(1, 2)
        cnn_features = []
        for conv in self.convs:
            y = F.relu(conv(cnn_input))
            cnn_features.append(F.adaptive_max_pool1d(y, 1).squeeze(-1))
            cnn_features.append(F.adaptive_avg_pool1d(y, 1).squeeze(-1))

        packed = pack_padded_sequence(embedded, lengths.detach().cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=input_ids.size(1))
        lengths_on_device = lengths.to(outputs.device)
        final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        rnn_features = [
            final_hidden,
            masked_mean_pool(outputs, lengths_on_device),
            masked_max_pool(outputs, lengths_on_device),
            self.attention_pool(outputs, lengths_on_device),
        ]

        features = torch.cat([*cnn_features, *rnn_features], dim=1)
        return self.classifier(self.norm(features))

### 6.5 Конфигурации экспериментов

Три эксперимента покрывают требования задания: только Conv1D, только рекуррентная модель и комбинированная архитектура.

In [ ]:
def build_cnn() -> nn.Module:
    return TextCNNMulti(vocab_size=len(vocab))


def build_rnn() -> nn.Module:
    return TextPackedRNN(vocab_size=len(vocab), rnn_type="gru")


def build_combined() -> nn.Module:
    return TextCNNBiGRU(vocab_size=len(vocab))


model_configs = [
    {
        "name": "TrainableCNN",
        "description": "Embedding + multi-kernel Conv1D + max/mean pooling",
        "factory": build_cnn,
        "loader_key": "TrainableCNN",
        "lr": 8e-4,
        "clip_grad": None,
    },
    {
        "name": "TrainablePackedBiGRU",
        "description": "Embedding + packed bidirectional GRU + final/mean/max/attention pooling",
        "factory": build_rnn,
        "loader_key": "TrainablePackedBiGRU",
        "lr": 6e-4,
        "clip_grad": GRAD_CLIP,
    },
    {
        "name": "TrainableCNNBiGRU",
        "description": "Parallel Conv1D branch + packed BiGRU branch with attention pooling",
        "factory": build_combined,
        "loader_key": "TrainableCNNBiGRU",
        "lr": 5e-4,
        "clip_grad": GRAD_CLIP,
    },
]

pd.DataFrame([{k: v for k, v in cfg.items() if k != "factory"} for cfg in model_configs])

Модели обучаются по `CrossEntropyLoss`. В этой итерации балансировка задается через `BALANCE_STRATEGY`: по умолчанию используется `WeightedRandomSampler`, а веса классов в loss выключены, чтобы не удваивать компенсацию дисбаланса. Early stopping и scheduler ориентируются на validation `macro-F1`, потому что accuracy плохо отражает качество на несбалансированных классах.

### 7.1 Метрики и class weights

Веса классов считаются по train-набору. Используется степень `0.25`, чтобы компенсировать дисбаланс мягко, не заставляя модель механически пере-предсказывать меньшие классы.

In [ ]:
def labels_from_ids(label_ids: np.ndarray | list[int]) -> list[str]:
    return [ID_TO_LABEL[int(label_id)] for label_id in label_ids]


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    precision, recall, f1_by_class, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    for idx, label in ID_TO_LABEL.items():
        metrics[f"{label}_precision"] = precision[idx]
        metrics[f"{label}_recall"] = recall[idx]
        metrics[f"{label}_f1"] = f1_by_class[idx]
    return metrics


class_counts = train_df[TARGET_COL].value_counts().reindex(LABEL_ORDER)
count_ratio = class_counts.sum() / (NUM_CLASSES * class_counts)
loss_weight_values = np.power(count_ratio.values.astype("float32"), CLASS_WEIGHT_ALPHA)

if BALANCE_STRATEGY == "loss_weights":
    class_weights = torch.tensor(loss_weight_values, dtype=torch.float32, device=DEVICE)
else:
    class_weights = None

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

pd.DataFrame({
    "class": LABEL_ORDER,
    "count": class_counts.values,
    "loss_weight_if_enabled": loss_weight_values,
    "active_balance_strategy": BALANCE_STRATEGY,
})

### 7.2 AMP и перенос batch на устройство

Вспомогательные функции скрывают различия между CUDA и CPU/MPS. На CUDA используется `torch.amp`.

In [ ]:
def autocast_context():
    if USE_AMP:
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return nullcontext()


def make_scaler():
    if USE_AMP:
        return torch.amp.GradScaler("cuda", enabled=True)
    return None


def move_batch(batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {
        key: value.to(DEVICE, non_blocking=PIN_MEMORY)
        for key, value in batch.items()
    }

### 7.3 Один проход обучения или оценки

Функция используется и для train, и для validation. Во время обучения работает AMP, gradient clipping и optimizer step.

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scaler=None,
    clip_grad: float | None = None,
) -> tuple[float, dict[str, float]]:
    is_train = optimizer is not None
    model.train(is_train)
    losses: list[float] = []
    all_preds: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []

    for batch in tqdm(loader, leave=False, desc="train" if is_train else "eval"):
        batch = move_batch(batch)
        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with autocast_context():
                logits = model(batch["input_ids"], batch["lengths"])
                loss = criterion(logits, batch["labels"])

            if is_train:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    if clip_grad is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if clip_grad is not None:
                        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                    optimizer.step()

        losses.append(float(loss.detach().cpu()))
        all_preds.append(logits.detach().argmax(dim=1).cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    return float(np.mean(losses)), compute_metrics(y_true, y_pred)

### 7.4 Оценка модели

`evaluate_model` возвращает logits, probabilities, predictions, labels и метрики. Bias можно передать после calibration.

In [ ]:
def apply_logit_bias(logits: torch.Tensor, bias: np.ndarray | None = None) -> torch.Tensor:
    if bias is None:
        return logits
    return logits + torch.tensor(bias, dtype=logits.dtype, device=logits.device).view(1, -1)


@torch.inference_mode()
def evaluate_model(model: nn.Module, loader: DataLoader, bias: np.ndarray | None = None) -> dict:
    model.eval()
    logits_list: list[torch.Tensor] = []
    labels_list: list[torch.Tensor] = []

    for batch in tqdm(loader, leave=False, desc="predict"):
        batch = move_batch(batch)
        logits = model(batch["input_ids"], batch["lengths"])
        logits_list.append(logits.detach().cpu())
        labels_list.append(batch["labels"].detach().cpu())

    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list).numpy()
    adjusted_logits = apply_logit_bias(logits, bias)
    probabilities = torch.softmax(adjusted_logits, dim=1).numpy()
    predictions = adjusted_logits.argmax(dim=1).numpy()

    return {
        "logits": logits,
        "probabilities": probabilities,
        "predictions": predictions,
        "labels": labels,
        "metrics": compute_metrics(labels, predictions),
    }

### 7.5 Optimizer и early stopping

Для всех моделей используется `AdamW`. Scheduler уменьшает learning rate, если validation `macro-F1` перестает расти.

In [ ]:
def make_optimizer(model: nn.Module, lr: float) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)


def train_model(model: nn.Module, config: dict, loaders: dict[str, DataLoader], criterion: nn.Module) -> dict:
    model = model.to(DEVICE)
    optimizer = make_optimizer(model, lr=config["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
    )
    scaler = make_scaler()

    best_state = deepcopy(model.state_dict())
    best_val_macro_f1 = -1.0
    best_epoch = 0
    wait = 0
    history: list[dict] = []
    start_time = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_metrics = run_epoch(
            model,
            loaders["train"],
            criterion,
            optimizer=optimizer,
            scaler=scaler,
            clip_grad=config.get("clip_grad"),
        )
        val_loss, val_metrics = run_epoch(model, loaders["val"], criterion)
        scheduler.step(val_metrics["macro_f1"])

        row = {
            "model": config["name"],
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_macro_f1": train_metrics["macro_f1"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_accuracy": val_metrics["accuracy"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(
            f"{config['name']} epoch {epoch}: "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}, val_acc={val_metrics['accuracy']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    train_time = time.perf_counter() - start_time
    model.load_state_dict(best_state)
    return {
        "model": model,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "train_time": train_time,
        "params": count_parameters(model),
    }

### 7.6 Logit calibration

Bias подбирается только на validation logits. Test не используется для настройки порогов или bias.

In [ ]:
def find_best_logit_bias(
    logits: torch.Tensor,
    labels: np.ndarray,
    grid: np.ndarray | None = None,
) -> tuple[np.ndarray, float]:
    if grid is None:
        grid = np.linspace(-1.2, 1.2, 25)
    best_bias = np.zeros(NUM_CLASSES, dtype="float32")
    best_score = -1.0

    for b_negative in grid:
        for b_neutral in grid:
            bias = np.array([b_negative, b_neutral, 0.0], dtype="float32")
            predictions = apply_logit_bias(logits, bias).argmax(dim=1).numpy()
            score = f1_score(labels, predictions, average="macro", zero_division=0)
            if score > best_score:
                best_score = score
                best_bias = bias

    return best_bias, best_score

### 7.7 Inference для новых текстов

Новые отзывы проходят ту же предобработку, токенизацию, head-tail truncation и calibration, что и test-набор.

In [ ]:
@torch.inference_mode()
def predict_texts(
    model: nn.Module,
    texts: list[str],
    vocab: dict[str, int],
    max_len: int,
    bias: np.ndarray | None = None,
) -> pd.DataFrame:
    cleaned = [preprocess_text(text, mode=PREPROCESS_MODE) for text in texts]
    input_ids, lengths = encode_texts(cleaned, vocab, max_len)
    model.eval()
    logits = model(input_ids.to(DEVICE), lengths.to(DEVICE)).detach().cpu()
    probabilities = torch.softmax(apply_logit_bias(logits, bias), dim=1).numpy()
    predictions = probabilities.argmax(axis=1)

    result = pd.DataFrame({
        "text": texts,
        "cleaned_text": cleaned,
        "predicted_label": labels_from_ids(predictions),
    })
    for idx, label in ID_TO_LABEL.items():
        result[f"p_{label}"] = probabilities[:, idx]
    return result

### 7.8 Обучение одной модели

Функция собирает обучение, raw validation, calibration и calibrated validation в один reproducible-блок.

In [ ]:
def train_and_calibrate(config: dict, seed: int = SEED) -> tuple[dict, dict, pd.DataFrame]:
    set_seed(seed)
    print(f"Training {config['name']} with seed={seed}")
    model = config["factory"]()
    loaders = loaders_by_model[config["loader_key"]]
    train_result = train_model(model, config, loaders, criterion)

    val_raw = evaluate_model(train_result["model"], loaders["val"])
    best_bias, calibrated_val_macro_f1 = find_best_logit_bias(val_raw["logits"], val_raw["labels"])
    val_calibrated = evaluate_model(train_result["model"], loaders["val"], bias=best_bias)

    row = {
        "model": config["name"],
        "seed": seed,
        "best_epoch": train_result["best_epoch"],
        "raw_val_macro_f1": val_raw["metrics"]["macro_f1"],
        "calibrated_val_macro_f1": val_calibrated["metrics"]["macro_f1"],
        "raw_val_accuracy": val_raw["metrics"]["accuracy"],
        "calibrated_val_accuracy": val_calibrated["metrics"]["accuracy"],
        "neutral_f1": val_calibrated["metrics"]["neutral_f1"],
        "train_time": train_result["train_time"],
        "params": train_result["params"],
    }
    bundle = {
        **train_result,
        "config": config,
        "seed": seed,
        "val_raw": val_raw,
        "val_calibrated": val_calibrated,
        "bias": best_bias,
        "calibrated_val_macro_f1": calibrated_val_macro_f1,
        "max_len": {"TrainableCNN": CNN_MAX_LEN, "TrainablePackedBiGRU": RNN_MAX_LEN, "TrainableCNNBiGRU": COMBINED_MAX_LEN}[config["loader_key"]],
    }
    history = train_result["history"].copy()
    history["seed"] = seed
    return bundle, row, history

### 7.9 Запуск трех экспериментов

Основной запуск обучает по одному seed для каждой архитектуры. Seed ensemble оставлен выключенным, чтобы базовый прогон не становился неожиданно долгим.

In [ ]:
trained_models = {}
validation_rows = []
history_frames = []

for config in model_configs:
    bundle, row, history = train_and_calibrate(config, seed=SEED)
    trained_models[config["name"]] = bundle
    validation_rows.append(row)
    history_frames.append(history)

validation_summary_df = pd.DataFrame(validation_rows).sort_values("calibrated_val_macro_f1", ascending=False)
history_df = pd.concat(history_frames, ignore_index=True)
display(validation_summary_df)

### 7.10 Опциональный seed ensemble

Если нужно выжать более стабильный результат, можно включить `RUN_SEED_ENSEMBLE=True` и обучить выбранные архитектуры на нескольких seed. По умолчанию этот блок ничего не запускает.

In [ ]:
ensemble_bundles = {}

if RUN_SEED_ENSEMBLE:
    for config in model_configs:
        seed_bundles = []
        for seed in ENSEMBLE_SEEDS:
            bundle, _, _ = train_and_calibrate(config, seed=seed)
            seed_bundles.append(bundle)
        ensemble_bundles[config["name"]] = seed_bundles
else:
    print("Seed ensemble выключен. Для запуска установите RUN_SEED_ENSEMBLE=True.")

### 7.11 Динамика validation macro-F1

График нужен для проверки, как быстро модели сходятся и где сработал early stopping.

In [ ]:
fig = px.line(
    history_df,
    x="epoch",
    y="val_macro_f1",
    color="model",
    markers=True,
    title="Динамика validation macro-F1",
    labels={"epoch": "Эпоха", "val_macro_f1": "Validation macro-F1"},
)
fig.update_layout(height=460)
fig.show()

### 7.12 Оценка на test

Test используется только после обучения и calibration. Для каждой модели показываются raw и calibrated метрики.

In [ ]:
def collect_test_results(model_name: str, bundle: dict) -> tuple[dict, dict]:
    loader = loaders_by_model[bundle["config"]["loader_key"]]["test"]
    raw = evaluate_model(bundle["model"], loader)
    calibrated = evaluate_model(bundle["model"], loader, bias=bundle["bias"])
    row = {
        "model": model_name,
        "raw_macro_f1": raw["metrics"]["macro_f1"],
        "calibrated_macro_f1": calibrated["metrics"]["macro_f1"],
        "raw_accuracy": raw["metrics"]["accuracy"],
        "calibrated_accuracy": calibrated["metrics"]["accuracy"],
        "weighted_f1": calibrated["metrics"]["weighted_f1"],
        "neutral_f1": calibrated["metrics"]["neutral_f1"],
        "train_time": bundle["train_time"],
        "params": bundle["params"],
    }
    return row, {"raw": raw, "calibrated": calibrated}


test_rows = []
test_predictions = {}
for model_name, bundle in trained_models.items():
    row, predictions = collect_test_results(model_name, bundle)
    test_rows.append(row)
    test_predictions[model_name] = predictions

test_summary_df = pd.DataFrame(test_rows).sort_values("calibrated_macro_f1", ascending=False)
best_model_name = validation_summary_df.iloc[0]["model"]
best_bundle = trained_models[best_model_name]
best_test = test_predictions[best_model_name]["calibrated"]

display(test_summary_df)
print(f"Best by calibrated validation macro-F1: {best_model_name}")

### 7.13 Classification report и confusion matrix

Для лучшей по validation модели показываем подробный отчет и матрицу ошибок на test.

In [ ]:
report_dict = classification_report(
    labels_from_ids(best_test["labels"]),
    labels_from_ids(best_test["predictions"]),
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
cm = confusion_matrix(best_test["labels"], best_test["predictions"], labels=list(range(NUM_CLASSES)))

print(classification_report(
    labels_from_ids(best_test["labels"]),
    labels_from_ids(best_test["predictions"]),
    labels=LABEL_ORDER,
    zero_division=0,
))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER)
plt.title(f"Confusion matrix: {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## 8. Анализ ошибок

Разбираем конкретные ошибки лучшей модели. Для отзывов Кинопоиска особенно ожидаемы ошибки между `neutral` и `positive`, потому что нейтральные рецензии часто содержат и похвалу, и критику.

### 8.1 Вспомогательные функции анализа

Сокращаем длинные отзывы, чтобы таблица ошибок оставалась читаемой.

In [ ]:
def shorten_text(text: str, max_chars: int = 320) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."

### 8.2 Примеры ошибок

Показываем 3-5 неверных классификаций с вероятностями классов. По ним удобнее понять, где модель путает тональность.

In [ ]:
analysis_df = test_df.copy()
analysis_df["true_label"] = labels_from_ids(best_test["labels"])
analysis_df["predicted_label"] = labels_from_ids(best_test["predictions"])
analysis_df["confidence"] = best_test["probabilities"].max(axis=1)
for idx, label in ID_TO_LABEL.items():
    analysis_df[f"p_{label}"] = best_test["probabilities"][:, idx]

errors_df = analysis_df[analysis_df["true_label"] != analysis_df["predicted_label"]].copy()
errors_df = errors_df.sort_values("confidence", ascending=False).head(5)
errors_df["review_short"] = errors_df[TEXT_COL].map(shorten_text)

display(errors_df[["true_label", "predicted_label", "confidence", "p_negative", "p_neutral", "p_positive", "review_short"]])

## 9. Проверка на новых текстах

Пять коротких отзывов проходят через тот же pipeline. Рядом указана ожидаемая метка, чтобы можно было быстро оценить поведение модели на простых примерах.

### 9.1 Новые отзывы

Отзывы подобраны так, чтобы покрыть явно положительную, явно отрицательную и смешанную тональность.

In [ ]:
new_examples = [
    {"text": "Фильм получился живым и смешным, актеры отлично держат темп.", "expected_label": "positive"},
    {"text": "Сюжет разваливается уже к середине, финал совсем не работает.", "expected_label": "negative"},
    {"text": "Нормальное кино на вечер: есть удачные сцены, но без сильного впечатления.", "expected_label": "neutral"},
    {"text": "Не шедевр, но персонажи понравились, а музыка прямо вытянула настроение.", "expected_label": "positive"},
    {"text": "Картинка красивая, зато диалоги скучные и слишком затянутые.", "expected_label": "neutral"},
]
new_examples_df = pd.DataFrame(new_examples)
display(new_examples_df)

### 9.2 Предсказания на новых отзывах

Используется лучшая модель и calibration bias, подобранный только на validation.

In [ ]:
new_predictions = predict_texts(
    best_bundle["model"],
    new_examples_df["text"].tolist(),
    vocab,
    max_len=best_bundle["max_len"],
    bias=best_bundle["bias"],
)
new_predictions.insert(1, "expected_label", new_examples_df["expected_label"])
display(new_predictions)

## 10. Выводы

Финальные выводы строятся после выполнения ноутбука: они привязаны к метрикам лучшей модели, структуре ошибок и сравнению архитектур.

### 10.1 Итоговая интерпретация

Эта ячейка формирует связный вывод по фактическим метрикам после запуска всех экспериментов.

In [ ]:
best_val_row = validation_summary_df[validation_summary_df["model"] == best_model_name].iloc[0]
best_test_row = test_summary_df[test_summary_df["model"] == best_model_name].iloc[0]

summary_text = f"""
Лучшей моделью по validation `macro-F1` стала `{best_model_name}`: после calibration она получила `macro-F1={best_val_row['calibrated_val_macro_f1']:.4f}` на validation и `macro-F1={best_test_row['calibrated_macro_f1']:.4f}` на test. Для этой задачи я смотрю именно на `macro-F1`, потому что accuracy может выглядеть приемлемо даже тогда, когда модель плохо распознает менее частые классы.

В этой итерации train, validation и test оставлены в естественном распределении классов, а частотные признаки строятся без утечки: rating-токены извлекаются только из текста `review`, частотные биграммы выбираются только по train-части, словарь строится только по `train_df`, calibration подбирается только на validation. Для борьбы с дисбалансом по умолчанию используется `WeightedRandomSampler`, а не одновременная комбинация sampler и class weights.

Главные изменения относительно предыдущего запуска: увеличена длина последовательностей, добавлены rating-токены, train-only биграммы и attention pooling в рекуррентных ветках. По матрице ошибок нужно отдельно смотреть пары `neutral ↔ positive` и `neutral ↔ negative`: нейтральные отзывы часто смешивают похвалу и критику, а разметка сводит этот баланс к одной метке. Если качество снова ограничится около того же уровня, следующий шаг уже вне базовых ограничений лабораторной — предобученные fastText/ruBERT-подобные представления.
"""
display(Markdown(summary_text))

### 10.2 Что можно улучшить дальше

Основной notebook оставлен строго в рамках задания: обучаемый `Embedding` поверх токенизированных последовательностей, Conv1D, GRU и комбинированная архитектура. Для дальнейшего улучшения качества можно попробовать предобученные эмбеддинги, например fastText-подобные решения для русского языка: они дают более сильные начальные векторы слов и могут особенно помочь редким словоформам и словам вне словаря. Я бы выносил такой эксперимент отдельно, чтобы не смешивать обязательную часть лабораторной с расширенным вариантом.